# Task 3: Quick analysis

Mean NDVI in 30-day windows before/after `sowing_date` by `crop_type`, using [cleaned_parcel_timeseries.csv](cleaned_parcel_timeseries.csv).

**Filter:** `sensor_status == OK` only (exclude ERROR and MISSING).

In [1]:
from pathlib import Path

import polars as pl

ROOT = Path(".")
TIMESERIES_PATH = ROOT / "cleaned_parcel_timeseries.csv"
WINDOW_DAYS = 30

In [2]:
df = pl.read_csv(
    TIMESERIES_PATH,
    schema_overrides={
        "parcel_id": pl.Utf8,
        "date": pl.Utf8,
        "sensor_status": pl.Utf8,
        "mill_id": pl.Utf8,
        "crop_type": pl.Utf8,
        "sowing_date": pl.Utf8,
    },
)

df = df.with_columns(
    pl.col("date").str.to_date("%Y-%m-%d", strict=False),
    pl.col("sowing_date").str.to_date("%Y-%m-%d", strict=False),
    pl.col("ndvi_value").cast(pl.Float64, strict=False),
    pl.col("temperature_c").cast(pl.Float64, strict=False),
    pl.col("rainfall_mm").cast(pl.Float64, strict=False),
    pl.col("area_hectares").cast(pl.Float64, strict=False),
)

print(f"Loaded: {df.height:,} rows x {df.width} cols")
print(df.select("crop_type").unique().sort("crop_type"))

Loaded: 3,399 rows x 19 cols
shape: (3, 1)
┌───────────┐
│ crop_type │
│ ---       │
│ str       │
╞═══════════╡
│ soybean   │
│ sugarcane │
│ wheat     │
└───────────┘


### Filter sensor status

In [3]:
print("sensor_status counts (all rows):")
print(df.group_by("sensor_status").len().sort("len", descending=True))

excluded = df.height - df.filter(pl.col("sensor_status") == "OK").height
analysis = (
    df.filter(pl.col("sensor_status") == "OK")
    .filter(pl.col("ndvi_value").is_not_null())
)
excluded_ndvi = df.filter(pl.col("sensor_status") == "OK").height - analysis.height

print(f"\nExcluded (not OK): {excluded:,} rows")
print(f"Excluded (OK but null ndvi_value): {excluded_ndvi:,} rows")
print(f"Analysis rows: {analysis.height:,}")

sensor_status counts (all rows):
shape: (3, 2)
┌───────────────┬──────┐
│ sensor_status ┆ len  │
│ ---           ┆ ---  │
│ str           ┆ u32  │
╞═══════════════╪══════╡
│ OK            ┆ 3017 │
│ ERROR         ┆ 245  │
│ MISSING       ┆ 137  │
└───────────────┴──────┘

Excluded (not OK): 382 rows
Excluded (OK but null ndvi_value): 0 rows
Analysis rows: 3,017


### NDVI windows around sowing

Half-open windows (sowing day excluded):

- **Before:** `[sowing_date - 30 days, sowing_date)`
- **After:** `(sowing_date, sowing_date + 30 days]`

In [4]:
windowed = analysis.with_columns(
    (pl.col("date") >= pl.col("sowing_date") - pl.duration(days=WINDOW_DAYS)).alias(
        "_in_before_range"
    ),
    (pl.col("date") < pl.col("sowing_date")).alias("_before"),
    (pl.col("date") > pl.col("sowing_date")).alias("_after_start"),
    (pl.col("date") <= pl.col("sowing_date") + pl.duration(days=WINDOW_DAYS)).alias(
        "_in_after_range"
    ),
).with_columns(
    (pl.col("_in_before_range") & pl.col("_before")).alias("is_before_window"),
    (pl.col("_after_start") & pl.col("_in_after_range")).alias("is_after_window"),
)

parcel_metrics = windowed.group_by("parcel_id", "crop_type", "sowing_date").agg(
    pl.col("ndvi_value").filter(pl.col("is_before_window")).mean().alias("mean_ndvi_before"),
    pl.col("ndvi_value").filter(pl.col("is_after_window")).mean().alias("mean_ndvi_after"),
    pl.col("is_before_window").sum().alias("n_readings_before"),
    pl.col("is_after_window").sum().alias("n_readings_after"),
)

print(f"Parcels with metrics: {parcel_metrics.height}")
display(parcel_metrics.head(5))

Parcels with metrics: 25


parcel_id,crop_type,sowing_date,mean_ndvi_before,mean_ndvi_after,n_readings_before,n_readings_after
str,str,date,f64,f64,u32,u32
"""PARCEL_025""","""sugarcane""",2026-02-07,0.185043,0.388455,23,22
"""PARCEL_006""","""sugarcane""",2026-02-11,0.16785,0.329957,20,23
"""PARCEL_019""","""sugarcane""",2026-01-18,0.179857,0.319962,14,26
"""PARCEL_014""","""sugarcane""",2026-01-05,0.184667,0.325818,3,22
"""PARCEL_015""","""sugarcane""",2026-01-06,0.241,0.37787,5,23


### Aggregate by crop_type

In [5]:
crop_summary = (
    parcel_metrics.group_by("crop_type")
    .agg(
        pl.col("mean_ndvi_before").mean().alias("mean_ndvi_before"),
        pl.col("mean_ndvi_after").mean().alias("mean_ndvi_after"),
        pl.col("parcel_id").n_unique().alias("n_parcels"),
    )
    .sort("crop_type")
)

crop_summary

crop_type,mean_ndvi_before,mean_ndvi_after,n_parcels
str,f64,f64,u32
"""soybean""",0.174625,0.313509,4
"""sugarcane""",0.176545,0.336198,19
"""wheat""",0.174491,0.311366,2


### Interpretation

After keeping only `sensor_status == OK` readings, mean NDVI rises in the 30 days after sowing for every crop type — roughly 0.17 before vs 0.31–0.34 after — which matches expected canopy growth once crops are established. Sugarcane shows the highest post-sowing NDVI (~0.34) and the largest sample (19 parcels), while soybean and wheat are similar to each other but based on only 4 and 2 parcels respectively, so those two estimates should be treated as indicative rather than definitive.